# SAKSHI — full pipeline run in Colab (D1 + D4 + D5 evidence)

Project rule: nothing executes on the developer laptop. This notebook clones the repo and runs, in order:
1. unit + chaos tests, 2. data download (MEVA CC BY 4.0, Commons), 3. clip preparation (DVR clock burn-in, red-team manipulations),
4. models (ONNX from the fine-tune notebook via Drive, HF weights), 5. edge pipeline on every video scenario,
6. robustness curves, 7. accuracy metrics + report. The last cell prints every result so it can be copied back into the repo.

Edge fps measured here is on Colab's CPU (no GPU used by the edge: ONNX Runtime CPU provider).

In [ ]:
!rm -rf /content/sakshi && git clone -q --depth 1 https://github.com/nevil-sanish/SIH26245.git /content/sakshi
%cd /content/sakshi
!pip install -q -r edge/requirements.txt 2>&1 | tail -3
!git log --oneline -1 && python --version && nproc && grep -m1 'model name' /proc/cpuinfo

In [ ]:
!python -m pytest -q tests 2>&1 | tail -5
!for m in integrity packet evidence; do python -m edge.$m; done

In [ ]:
!python data/download.py
!python -m redteam.runner prepare > /tmp/prepare.json && ls -la data/work/clips

In [ ]:
# ONNX detectors from the fine-tune notebook (saved to MyDrive/sakshi_models), then HF weights.
from google.colab import drive
drive.mount('/content/drive')
!python models/download.py --onnx-from /content/drive/MyDrive/sakshi_models --write-checksums
!ls -la models

In [ ]:
!rm -rf data/edge_state && python -m redteam.runner video 2>&1 | tail -40

In [ ]:
!python -m eval.robustness 2>&1 | grep -v Warning | tail -8

In [ ]:
!python -m eval.metrics --seeds 20 && python -m eval.report

In [ ]:
# Results back to the repo: copy these files (also saved to Drive).
!mkdir -p /content/drive/MyDrive/sakshi_results && cp data/work/*.json docs/ACCURACY_REPORT.md models/checksums.txt /content/drive/MyDrive/sakshi_results/
!cd data && tar czf /content/drive/MyDrive/sakshi_results/edge_state.tgz edge_state
for f in ['docs/ACCURACY_REPORT.md', 'models/checksums.txt', 'data/work/accuracy.json', 'data/work/robustness.json', 'data/work/edge_runs.json']:
    print(f'===== BEGIN {f} =====')
    print(open(f).read())
    print(f'===== END {f} =====')